In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = '/content/drive/MyDrive/final_project/df_clean_base.csv'
SAVE_PATH = '/content/drive/MyDrive/final_project/df_processed.csv'

df = pd.read_csv(FILE_PATH)
print("Shape:", df.shape)
print("Missing:", df.isnull().sum().sum())
print("\nSample description:")
print(df['ticket_description'].iloc[0][:150])

Mounted at /content/drive
Shape: (8469, 20)
Missing: 19919

Sample description:
I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website ad


In [2]:
!pip install spacy textblob -q
!python -m spacy download en_core_web_sm -q

import spacy
from textblob import TextBlob
from tqdm.notebook import tqdm
tqdm.pandas()

nlp = spacy.load('en_core_web_sm')
print("spaCy and TextBlob ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy and TextBlob ready!


In [3]:
# BAsic text cleaning
def clean_text(text):
    """
    Basic cleaning:
    - Lowercase
    - Remove HTML tags
    - Remove URLs
    - Remove punctuation and numbers
    - Strip extra whitespace
    """
    if pd.isnull(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'<.*?>',       ' ', text)   # HTML tags
    text = re.sub(r'http\S+',     ' ', text)   # URLs
    text = re.sub(r'[^a-z\s]',   ' ', text)   # punctuation/numbers
    text = re.sub(r'\s+',         ' ', text)   # extra spaces
    return text.strip()

# Test on real data
sample = df['ticket_description'].iloc[0]
print("ORIGINAL :", sample[:150])
print("\nCLEANED  :", clean_text(sample)[:150])



ORIGINAL : I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website ad

CLEANED  : i m having an issue with the product purchased please assist your billing zip code is we appreciate that you have requested a website address please d


In [4]:
df['cleaned_description'] = df['ticket_description'].apply(clean_text)
display(df[['ticket_description', 'cleaned_description']].head())

,ticket_description,cleaned_description
0,I'm having an issue with the {product_purchase...,i m having an issue with the product purchased...
1,I'm having an issue with the {product_purchase...,i m having an issue with the product purchased...
2,I'm facing a problem with my {product_purchase...,i m facing a problem with my product purchased...
3,I'm having an issue with the {product_purchase...,i m having an issue with the product purchased...
4,I'm having an issue with the {product_purchase...,i m having an issue with the product purchased...


In [5]:
#NLP preprocessing function
def preprocess_text(text):
    """
    Full NLP pipeline using spaCy:
    - Clean text
    - Tokenize
    - Remove stopwords
    - Remove short tokens
    - Lemmatize
    """
    text = clean_text(text)
    if text == '':
        return ''
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.text) > 2
    ]
    return ' '.join(tokens)

# Test on real data
sample = df['ticket_description'].iloc[0]
print("ORIGINAL    :", sample[:150])
print("\nPROCESSED   :", preprocess_text(sample)[:150])

ORIGINAL    : I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website ad

PROCESSED   : have issue product purchase assist billing zip code appreciate request website address double check email address try troubleshoot step mention user m


In [6]:
# Apply preprocessing to full dataset
# Combine subject + description
df['combined_text'] = (
    df['ticket_subject'].fillna('') + ' ' +
    df['ticket_description'].fillna('')
)

# Basic cleaning (fast)
print("Step 1: Applying basic cleaning...")
df['text_cleaned'] = df['combined_text'].apply(clean_text)

# Full NLP preprocessing (slower)
print("Step 2: Applying NLP preprocessing...")
df['text_processed'] = df['text_cleaned'].progress_apply(preprocess_text)

print("\nDone! Sample comparison:")
print("Cleaned  :", df['text_cleaned'].iloc[0][:100])
print("Processed:", df['text_processed'].iloc[0][:100])

Step 1: Applying basic cleaning...
Step 2: Applying NLP preprocessing...


  0%|          | 0/8469 [00:00<?, ?it/s]


Done! Sample comparison:
Cleaned  : product setup i m having an issue with the product purchased please assist your billing zip code is 
Processed: product setup have issue product purchase assist billing zip code appreciate request website address


In [7]:
#Text-based numeric features

def get_sentiment(text):
    try:
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0.0

# Word count
df['word_count']      = df['text_cleaned'].apply(lambda x: len(str(x).split()))

# Char count
df['char_count']      = df['text_cleaned'].apply(lambda x: len(str(x)))

# Sentiment polarity
print("Computing sentiment scores...")
df['sentiment_score'] = df['text_cleaned'].progress_apply(get_sentiment)

print("\nText features summary:")
print(df[['word_count', 'char_count', 'sentiment_score']].describe().round(2))

Computing sentiment scores...


  0%|          | 0/8469 [00:00<?, ?it/s]


Text features summary:
       word_count  char_count  sentiment_score
count     8469.00     8469.00          8469.00
mean        52.20      290.14             0.06
std          8.95       46.55             0.20
min         26.00      156.00            -0.70
25%         48.00      269.00            -0.03
50%         55.00      301.00             0.02
75%         59.00      322.00             0.15
max         68.00      399.00             1.00


In [8]:
# TF-IDF vectors (for baseline ML)
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import os

os.makedirs('/content/drive/MyDrive/final_project/models/', exist_ok=True)
MODEL_DIR = '/content/drive/MyDrive/final_project/models/'

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df['text_processed'])

# Save vectorizer for reuse in modelling
joblib.dump(tfidf, MODEL_DIR + 'tfidf_vectorizer.pkl')

print("TF-IDF matrix shape :", tfidf_matrix.shape)
print("Top 20 features     :", tfidf.get_feature_names_out()[:20].tolist())
print("Vectorizer saved to Drive!")

TF-IDF matrix shape : (8469, 5000)
Top 20 features     : ['ability', 'able', 'able accept', 'able add', 'able find', 'able fix', 'able help', 'able purchase', 'able sell', 'able use', 'absolutely', 'accept', 'accept payment', 'access', 'access accidentally', 'access account', 'access encounter', 'access face', 'access forget', 'access glitch']
Vectorizer saved to Drive!


In [9]:
# Tabuar engineering
# Encode categorical columns
df['channel_encoded'] = df['ticket_channel'].astype('category').cat.codes
df['gender_encoded']  = df['customer_gender'].astype('category').cat.codes
df['product_encoded'] = df['product_purchased'].astype('category').cat.codes

# Ticket age — days since purchase
df['date_of_purchase'] = pd.to_datetime(df['date_of_purchase'], errors='coerce')
reference_date         = pd.Timestamp('2024-01-01')
df['ticket_age_days']  = (reference_date - df['date_of_purchase']).dt.days.fillna(0).astype(int)

tabular_features = [
    'customer_age', 'channel_encoded', 'gender_encoded',
    'product_encoded', 'ticket_age_days',
    'word_count', 'char_count', 'sentiment_score'
]

print("Tabular features ready:")
print(df[tabular_features].describe().round(2))

Tabular features ready:
       customer_age  channel_encoded  gender_encoded  product_encoded  \
count       8469.00          8469.00         8469.00          8469.00   
mean          44.03             1.51            0.98            20.54   
std           15.30             1.11            0.81            12.12   
min           18.00             0.00            0.00             0.00   
25%           31.00             1.00            0.00            10.00   
50%           44.00             2.00            1.00            20.00   
75%           57.00             3.00            2.00            31.00   
max           70.00             3.00            2.00            41.00   

       ticket_age_days  word_count  char_count  sentiment_score  
count          8469.00     8469.00     8469.00          8469.00  
mean           1096.93       52.20      290.14             0.06  
std             211.24        8.95       46.55             0.20  
min             732.00       26.00      156.00        

In [10]:
# Encode + Train / Val / Test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode targets
le_type     = LabelEncoder()
le_priority = LabelEncoder()

df['ticket_type_encoded']     = le_type.fit_transform(df['ticket_type'])
df['ticket_priority_encoded'] = le_priority.fit_transform(df['ticket_priority'])

# Save encoders
joblib.dump(le_type,     MODEL_DIR + 'le_type.pkl')
joblib.dump(le_priority, MODEL_DIR + 'le_priority.pkl')

print("Ticket Type classes    :", le_type.classes_.tolist())
print("Ticket Priority classes:", le_priority.classes_.tolist())

# Stratified split
train_df, temp_df = train_test_split(
    df, test_size=0.3,
    stratify=df['ticket_type_encoded'],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    stratify=temp_df['ticket_type_encoded'],
    random_state=42
)

print(f"\nTrain : {len(train_df):,} rows ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val   : {len(val_df):,} rows ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test  : {len(test_df):,} rows ({len(test_df)/len(df)*100:.1f}%)")

Ticket Type classes    : ['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']
Ticket Priority classes: ['Critical', 'High', 'Low', 'Medium']

Train : 5,928 rows (70.0%)
Val   : 1,270 rows (15.0%)
Test  : 1,271 rows (15.0%)


In [11]:
# Save processed data to Drive
BASE = '/content/drive/MyDrive/final_project/'

df.to_csv(       BASE + 'df_processed.csv', index=False)
train_df.to_csv( BASE + 'df_train.csv',     index=False)
val_df.to_csv(   BASE + 'df_val.csv',       index=False)
test_df.to_csv(  BASE + 'df_test.csv',      index=False)

print("All files saved to Drive!")
print(f"  df_processed.csv — {len(df):,} rows, {df.shape[1]} columns")
print(f"  df_train.csv     — {len(train_df):,} rows")
print(f"  df_val.csv       — {len(val_df):,} rows")
print(f"  df_test.csv      — {len(test_df):,} rows")
print(f"\nNew columns added:")
new_cols = ['combined_text','text_cleaned','text_processed',
            'word_count','char_count','sentiment_score',
            'channel_encoded','gender_encoded','product_encoded',
            'ticket_age_days','ticket_type_encoded','ticket_priority_encoded']
for c in new_cols:
    print(f"  + {c}")


All files saved to Drive!
  df_processed.csv — 8,469 rows, 31 columns
  df_train.csv     — 5,928 rows
  df_val.csv       — 1,270 rows
  df_test.csv      — 1,271 rows

New columns added:
  + combined_text
  + text_cleaned
  + text_processed
  + word_count
  + char_count
  + sentiment_score
  + channel_encoded
  + gender_encoded
  + product_encoded
  + ticket_age_days
  + ticket_type_encoded
  + ticket_priority_encoded


In [12]:
df.shape

(8469, 31)